### Install Dependencies

In [ ]:
# Install core libraries
%pip install torch torchaudio librosa faiss-cpu pandas numpy laion-clap torchlibrosa torchvision flask

In [6]:
import requests
import os
from tqdm import tqdm

def download_file(url, filename):
    # Streaming the request allows us to download large files without memory issues
    response = requests.get(url, stream=True)
    
    # Get the file size from headers
    total_size = int(response.headers.get('content-length', 0))
    
    # tqdm creates a progress bar
    with open(filename, 'wb') as file, tqdm(
        desc=filename,
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)

# # Direct URL to the HTSAT-base model file
# URL = "https://huggingface.co/lukewys/laion_clap/resolve/main/630k-audioset-best.pt"
# LOCAL_FILE = "630k-audioset-best.pt"
# URL and target filename
URL = "https://huggingface.co/lukewys/laion_clap/resolve/main/music_speech_audioset_epoch_15_esc_89.98.pt"
LOCAL_FILE = "music_speech_audioset.pt"
if not os.path.exists(LOCAL_FILE):
    print("Starting download...")
    download_file(URL, LOCAL_FILE)
    print(f"Download complete: {LOCAL_FILE}")
else:
    print(f"File {LOCAL_FILE} already exists.")

Starting download...


Starting download...


music_speech_audioset.pt: 100%|██████████| 2.19G/2.19G [13:47<00:00, 2.84MiB/s]  

Starting download...


music_speech_audioset.pt: 100%|██████████| 2.19G/2.19G [13:47<00:00, 2.84MiB/s]  

Download complete: music_speech_audioset.pt


## Setup

In [2]:
# ----------------------
# Import from audio_indexing module
# ----------------------
from audio_indexing import (
    init_json_store,
    save_segments_to_json,
    load_audio,
    split_audio_into_segments,
    get_audio_embedding,
    get_text_embedding,
    index_audio_file,
    semantic_search_audio,
    semantic_search_audio_by_file,
    list_indexed_files,
    clear_index,
    remove_audio_file
)

c:\Users\gohyi\OneDrive\Documents\NTU\Year_2_Sem_2_THU\Web_Retrieval\project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\gohyi\OneDrive\Documents\NTU\Year_2_Sem_2_THU\Web_Retrieval\project\.venv\Lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11454.92it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias         

Load the specified checkpoint music_speech_audioset.pt from users.
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.weight 	 Lo

## Test Functions

#### Test embeddings

In [4]:
import numpy as np
# Test that identical audio segments return near-perfect similarity
test_audio = load_audio("fold1/72579-3-0-0.wav")
segments, _ = split_audio_into_segments(test_audio)
emb1 = get_audio_embedding(segments[0])
emb2 = get_audio_embedding(segments[0])  # Same segment
 
# Calculate cosine similarity manually
similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
print(f"Self-similarity score: {similarity:.3f}")  # Should be >0.95, proving embeddings work correctly

# test text embedding similarity with audio embedding
text_query = "dog barking loudly"
text_emb = get_text_embedding(text_query)
similarity = np.dot(emb1, text_emb) / (np.linalg.norm(emb1) * np.linalg.norm(text_emb))
print(f"Similarity between audio segment and text query '{text_query}': {similarity:.3f}")

full_audio_embedding = get_audio_embedding(test_audio)
similarity = np.dot(full_audio_embedding, text_emb) / (np.linalg.norm(full_audio_embedding) * np.linalg.norm(text_emb))
print(f"Similarity between full audio and text query '{text_query}': {similarity:.3f}")


Self-similarity score: 1.000
Similarity between audio segment and text query 'dog barking loudly': 0.214
Similarity between full audio and text query 'dog barking loudly': 0.257


#### Test indexing

In [ ]:
# Index any audio file (supports WAV, MP3, FLAC, etc.)
index_audio_file(r"audio\72579-3-0-0.wav")

✅ Audio file fold1\72579-3-0-0.wav is already indexed.


#### Test text searching

In [7]:
# Search for any sound description (no predefined tags required!)
results = semantic_search_audio("dog barking loudly", top_k=10)
 
print("Search Results:")
for res in results:
    print(f"File: {res['audio_file']} | Time: {res['start_time']}s - {res['end_time']}s | Similarity: {res['similarity_score']}")

Search Results:
File: fold1\193394-3-0-11.wav | Time: 0.5s - 2.0s | Similarity: 0.451
File: fold1\193394-3-0-4.wav | Time: 0.5s - 3.0s | Similarity: 0.451
File: fold1\101415-3-0-2.wav | Time: 0.0s - 3.0s | Similarity: 0.428
File: fold1\101415-3-0-8.wav | Time: 0.0s - 3.5s | Similarity: 0.409
File: fold1\193394-3-0-7.wav | Time: 0.0s - 3.5s | Similarity: 0.403
File: fold1\125791-3-0-13.wav | Time: 0.0s - 4.0s | Similarity: 0.398
File: fold1\54858-3-1-2.wav | Time: 3.0s - 4.0s | Similarity: 0.398
File: fold1\203356-3-0-1.wav | Time: 1.0s - 2.5s | Similarity: 0.393
File: fold1\72261-3-0-27.wav | Time: 2.0s - 4.0s | Similarity: 0.393
File: fold1\101415-3-0-3.wav | Time: 0.0s - 1.0s | Similarity: 0.39


In [18]:
# Common environmental sounds
print(semantic_search_audio("gunshot"))
print(semantic_search_audio("police siren"))
print(semantic_search_audio("glass breaking"))
print(semantic_search_audio("rain falling on roof"))

# Nuanced queries
print(semantic_search_audio("high pitched dog bark"))
print(semantic_search_audio("crowd cheering at a sports game"))
print(semantic_search_audio("car engine revving loudly"))
print(semantic_search_audio("baby crying softly"))

[{'audio_file': 'fold1/7061-6-0-0.wav', 'start_time': 0.0, 'end_time': 2.0, 'similarity_score': 0.29}, {'audio_file': 'fold1/72579-3-0-0.wav', 'start_time': 0.0, 'end_time': 4.0, 'similarity_score': 0.112}]
[{'audio_file': 'fold1/72579-3-0-0.wav', 'start_time': 1.0, 'end_time': 3.0, 'similarity_score': 0.015}]
[{'audio_file': 'fold1/7061-6-0-0.wav', 'start_time': 0.0, 'end_time': 2.0, 'similarity_score': 0.152}, {'audio_file': 'fold1/72579-3-0-0.wav', 'start_time': 2.0, 'end_time': 3.0, 'similarity_score': 0.012}]
[{'audio_file': 'fold1/7061-6-0-0.wav', 'start_time': 1.0, 'end_time': 2.0, 'similarity_score': 0.056}]
[{'audio_file': 'fold1/72579-3-0-0.wav', 'start_time': 0.0, 'end_time': 4.0, 'similarity_score': 0.334}, {'audio_file': 'fold1/7061-6-0-0.wav', 'start_time': 0.5, 'end_time': 1.5, 'similarity_score': 0.014}]
[]
[{'audio_file': 'fold1/7061-6-0-0.wav', 'start_time': 0.0, 'end_time': 2.0, 'similarity_score': 0.156}, {'audio_file': 'fold1/72579-3-0-0.wav', 'start_time': 0.5, 'e

#### Test audio searching

In [ ]:
semantic_search_audio_by_file("audio/72579-3-0-0.wav")


Query audio segmented into 7 segments


[{'audio_file': 'fold1\\72579-3-0-0.wav',
  'aggregated_score': 1.0,
  'num_matched_segments': 7,
  'segment_scores': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
  'best_matches': [{'query_segment_idx': 0,
    'query_time': [0.0, 1.0],
    'similarity_score': 1.0,
    'indexed_start': 0.0,
    'indexed_end': 1.0},
   {'query_segment_idx': 1,
    'query_time': [0.5, 1.5],
    'similarity_score': 1.0,
    'indexed_start': 0.5,
    'indexed_end': 1.5},
   {'query_segment_idx': 2,
    'query_time': [1.0, 2.0],
    'similarity_score': 1.0,
    'indexed_start': 1.0,
    'indexed_end': 2.0},
   {'query_segment_idx': 3,
    'query_time': [1.5, 2.5],
    'similarity_score': 1.0,
    'indexed_start': 1.5,
    'indexed_end': 2.5},
   {'query_segment_idx': 4,
    'query_time': [2.0, 3.0],
    'similarity_score': 1.0,
    'indexed_start': 2.0,
    'indexed_end': 3.0},
   {'query_segment_idx': 5,
    'query_time': [2.5, 3.5],
    'similarity_score': 1.0,
    'indexed_start': 2.5,
    'indexed_end': 3.5},
 

## Index all audio files in the folder

In [ ]:
from tqdm import tqdm
audio_folder = "audio"
audio_extensions = (".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac")

audio_files = sorted(
    os.path.join(audio_folder, f)
    for f in os.listdir(audio_folder)
    if f.lower().endswith(audio_extensions)
)

for audio_file in tqdm(audio_files):
    index_audio_file(audio_file, verbose=False)  # Set verbose to False for batch processing

100%|██████████| 873/873 [10:33<00:00,  1.38it/s]
